# Understanding the Open Econ Agent simulation

This notebook walks through a **small, laptop-friendly run** of the macroeconomic
agent simulation and explains what happens at each step.

**The idea.** Each simulated agent is a person. Every month, they read a description of
their situation — their job and expected wage, recent consumption, the taxes they paid and
the redistribution credit they received, current goods prices, their savings and the interest
rate — and they decide two things:

- **propensity to work** (0 = don't work this month, 1 = work full hours), and
- **propensity to consume** (what fraction of savings + income to spend on essential goods).

Those two decisions come from a **local open LLM** served by [Ollama](https://ollama.com).
The [Foundation / AI-Economist](https://github.com/salesforce/ai-economist) engine then steps the
economy forward — wages, prices, taxes, unemployment — and the next month begins.

We'll: configure a model → run ~5 agents for ~12 months → read one agent's actual dialog →
plot the resulting economy.

## 1. Setup

Run this from inside the repo. The cell below moves the working directory to the repo root
(so `config.yaml` and `ai_economist/` resolve) and points at a local Ollama model.

Make sure Ollama is running and the model is pulled first:

```bash
ollama pull llama3.1:8b
```

In [ ]:
import os

# Move to repo root if we're inside notebooks/
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print('Working dir:', os.getcwd())

# Which local model to use. Change this to any model you've pulled in Ollama.
os.environ['OLLAMA_MODEL'] = 'llama3.1:8b'
os.environ.setdefault('OLLAMA_BASE_URL', 'http://localhost:11434/v1')

## 2. Preflight: is the model available?

`check_ollama_model()` queries Ollama and fails with a clear message if the server is down
or the model isn't pulled — so we find out *now*, not halfway through a run.

In [ ]:
import importlib, simulate_utils
importlib.reload(simulate_utils)  # pick up the OLLAMA_MODEL we just set
simulate_utils.check_ollama_model()

## 3. Run a short simulation

We keep it tiny so it finishes on a laptop in a few minutes: **5 agents, 12 months**.
(The paper uses 100 agents × 240 months.)

Outputs are written under `data/<tag>/`: per-agent dialog transcripts, pickled snapshots
of the environment, a dense log of the whole run, and `run_meta.json` with the model, seed,
and how many responses failed to parse (`llm_errors`).

In [ ]:
from simulate import main

main(policy_model='llama', num_agents=5, episode_length=12, seed=0)

## 4. Find this run's output folder

In [ ]:
import glob
run_dir = max(glob.glob('data/llama-*'), key=os.path.getmtime)
print('Latest run:', run_dir)

import json
print(json.load(open(os.path.join(run_dir, 'run_meta.json'))))

## 5. Read one agent's dialog

Each agent has a transcript file under `dialogs/` named after the person. It contains the
**exact prompt the model saw** (persona + monthly observation + the JSON format request) and
the **decision it returned**. This is the single most useful thing to inspect: it shows *why*
an agent chose to work or spend the way it did.

A high `llm_errors` count in the metadata above means the model often returned something we
couldn't parse into `[work, consumption]`; in that case the agent fell back to a neutral
action `[1, 0.5]`. If that happens a lot, try a larger / more instruction-tuned model.

In [ ]:
dialog_files = sorted(glob.glob(os.path.join(run_dir, 'dialogs', '*')))
print('Agents:', [os.path.basename(f) for f in dialog_files])

# Print the first ~2000 characters of the first agent's transcript
with open(dialog_files[0]) as f:
    text = f.read()
print(text[:2000])

## 6. Plot the resulting economy

The `dense_log` records the world and every agent's state at each timestep. We'll look at:

1. **Average price** of essential goods over time (inflation / deflation).
2. **Average labor** supplied per agent each month (how much people chose to work).
3. The **final wealth distribution** across agents, summarized by a **Gini coefficient**
   (0 = perfectly equal, 1 = maximally unequal).

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt

dense = pickle.load(open(os.path.join(run_dir, 'dense_log.pkl'), 'rb'))
states = dense['states']
world = dense['world']
n_agents = len([k for k in states[-1] if k != 'p'])

# Average price per step
prices = [w.get('Price', np.nan) for w in world]

# Average labor per agent per step (endogenous 'Labor', 0..168)
avg_labor = []
for st in states:
    vals = [st[str(i)]['endogenous'].get('Labor', 0) for i in range(n_agents)]
    avg_labor.append(np.mean(vals))

# Final wealth per agent
final_wealth = np.array([states[-1][str(i)]['inventory']['Coin'] for i in range(n_agents)])

def gini(x):
    x = np.sort(np.asarray(x, dtype=float))
    n = len(x)
    if n == 0 or x.sum() == 0:
        return 0.0
    cum = np.cumsum(x)
    return (n + 1 - 2 * np.sum(cum) / cum[-1]) / n

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(prices, marker='o'); ax[0].set_title('Average goods price'); ax[0].set_xlabel('month')
ax[1].plot(avg_labor, marker='o', color='tab:green'); ax[1].set_title('Average labor per agent'); ax[1].set_xlabel('month')
ax[2].bar(range(n_agents), np.sort(final_wealth), color='tab:orange')
ax[2].set_title(f'Final wealth by agent (Gini = {gini(final_wealth):.2f})'); ax[2].set_xlabel('agent (sorted)')
plt.tight_layout(); plt.show()

## What to try next

- **Swap the model:** set `os.environ['OLLAMA_MODEL']` to another model you've pulled
  (e.g. `qwen2.5:7b`, `mistral`) and re-run — watch how `llm_errors` and the decisions change.
- **Scale up:** increase `num_agents` / `episode_length` (it gets slower).
- **Compare to the no-LLM baseline:** run `main(policy_model='complex', num_agents=5, episode_length=12)`
  and plot it the same way to see how rule-of-thumb agents differ from LLM agents.